In [3]:
### Import Functions

%run Packages.ipynb
%run Graph_Generators.ipynb
%run Cycle_Mtx_functions.ipynb
%run Metric_Repair_Algorithms.ipynb
# from random import sample


In [30]:
def verifier(G,S):
    """Verify if S is a light cover of G. If so, return a valid solution."""
    H = G.copy()
    G1=G.copy()
    M = max(G.edge_labels())
    for e in S:
        H.set_edge_label(e[0],e[1],M)
    D = H.distance_all_pairs(by_weight=True)
    for u,v,w in H.edges(sort=True):
        if D[u][v] != w:
            H.set_edge_label(u,v,D[u][v])
            if (u,v) not in S and (v,u) not in S:
                return 0
    return 1
    
        
    ## change every edge's weight to be the distance between endpoints
    ## if only edges in S were changed (increased for IOMR), return True, else False

def is_broken_cycle(G,C,light_edges=False):
    W = get_weights(G)
    """ G is a grpah, C is a list of edges corresponding to a cycle in G. Returns True iff C is broken.
    If light_edges = True, returns the list of light edges of C"""
    L = list(map(lambda x: W[x], C)) 
    i = np.argmax(L)
    C_light = [e for e in C if e!=C[i]] # collection of possible light edges in C
    if W[C[i]] <= sum([W[e] for e in C_light]):
        return False
    elif light_edges:
        return C_light
    else:
        return True

def brute_force_IOMR(G, support=False):
    """An awful algorithm. Basically solve hitting set argument to find optimal solution for IOMR.
    Support is for returning the actual support of a solution"""
    Light_Cycles= list()
    W = get_weights(G)
    H = G.copy()
    for e_w in H.edges(sort=True):
        e = (e_w[0],e_w[1]) #discard the weight
        for C in cycles_of_e(H,e,induced=False): #Iterate over all cycles in H containing e
            C_light = is_broken_cycle(G,C,light_edges=True) #will be a nonempty list of light edges if broken, False if not
            if C_light:
                Light_Cycles.append(set(C_light)) #Add the light edges as a set to hit
                
        H.delete_edge(e) # We've gone over all cycles containing e, either as light edge or heavy
    ### We now "simply" need to solve a hitting set problem over Light_Cycles. Utilize minihit.
    T = minihit.HsDag(Light_Cycles)
    T.solve(prune=True, sort=False)
    try:
        HS = next(T.generate_minimal_hitting_sets())
        return HS
    except StopIteration:
        return []

### Heuristics ###

def Gilbert_Jain_IOMR(Kn):
    """The Gilbert & Lalit heuristic - just arbitrarily pick and edge to fix in a broken triangle.
    Assumes input is Complete."""
    H = Kn.copy()
    M = H.weighted_adjacency_matrix()
    n = M.nrows()
    S = set()
    for k in range(n):
        for i in range(n):
            for j in range(i):
                if M[i,j] > M[i,k] + M[k,j]:
                    M[i,k] = M[i,j] - M[k,j]
                    S.add(tuple(sorted((i,k))))

    return S



def MVD_Pivot_Rec(ind,x,S,Kn):
    """I will think of x as the symmetric adjacency matrix and M as the fixed copy of the 
     adjacency matrix which we augment along the way"""
    if len(ind) <=2:
        return
    i = np.random.choice(ind)
    
    ind_i = ind.copy() #make sure we don't change the input
    ind_i.remove(i) # Remove the current pivot
    
    for j,k in list(combinations(ind_i,2)): #iterate over all triangles i,j,k
        ADD = False #flag - do we change the edge?
        if x[j,k] > x[i,j] + x[i,k]:
            x[j,k] = x[i,j] + x[i,k]
            ADD=True
        if x[j,k] < np.abs(x[i,j] - x[i,k]):
            x[j,k] = np.abs(x[i,j] - x[i,k])
            ADD=True
        if ADD:
            S.add(tuple(sorted((j,k))))
            
    MVD_Pivot_Rec(ind_i,x,S,Kn)
    return
        
def MVD_Pivot(Kn):
    """The algorithm from Fitting Metrics with Minimum Disagreement. This one solves a general MR, not IOMR"""
    A = Kn.weighted_adjacency_matrix()
    S = set()
    MVD_Pivot_Rec(list(range(Kn.num_verts())),A,S,Kn)
    T = np.triu(A)
    return S


def reduce_solution(S,G,Diff = []):
    """Takes a graph G and an extended HS S and reduces it to a HS for G (discards edges that are not in G).
    Diff is a matrix that allows to check if an edge was actually changed if needed"""
#     print(len(S))
    reduced_S = set()
    for e in G.edges(labels=False, sort=True): #check all edges
        if e in S:
            reduced_S.add(e) #if it hits a broken cycle, add it
    return reduced_S
        
def left_edge_heuristic(G):
    Kn = complete(G)
    S = Gilbert_Jain_IOMR(Kn)
    return reduce_solution(S,G)
    
def pivot_heuristic(G):
    Kn = complete(G)
    S = MVD_Pivot(Kn)
    return reduce_solution(S,G)


#### This is basically done now, just need to run some tests and then actually run it.
def l1_min_heuristic(G):
    """Solves metric repair by creating an l1 minimization problem. That is:
    1. Complete the graph G, obtain Gc
    2. Create the metric constraint matrix Phi, which checks all triangle inequalities in Gc
    3. Solve L1 minimization, guaranteed to satisfy all triangle inequalities
    4. Take the intersection of E(G) with the support of the solution. Since no broken triangles remain, that means
    no broken cycles in general remain, so the intersection must be a light cover of G"""
    
    ## Preprocess - comlplete the graph and get index dictionaries
    Gc = complete(G)
    D = make_index_encoding(Gc)
    w = get_weights_vector(Gc,D)
    
    ## Set up the minimization problem
    phi = cycle_matrix(Gc)
    Phi = np.transpose(phi)
    m,n = Phi.shape
    b = np.transpose(Phi) @ w
    c = np.ones(m)
    con = -np.transpose(Phi)
    
    ## Solve minimization problem
    soln = linprog(c,A_ub = con, b_ub = b, method = 'simplex') 
    
    ## Retrieve the support of solution
    S = set()
    for e in G.edges(sort=True):
        if soln.x[D[(e[0],e[1])]] > 0:
            S.add((e[0],e[1]))
    return S


def DOMR(G):
    # print("###################got here1")
    S = set()
    apsp = G.distance_all_pairs(by_weight=True)
    # print("###################got here2")
    for e in G.edges(sort=True):
        # print("###################got here3")
        if e[2] != apsp[e[0]][e[1]]:
            S.add((e[0],e[1]))
    return(S)
            

In [17]:
### Rishi Bad example constructor


# n=10
# d = [2**i for i in range(n)]
# D = np.zeros((n,n))
# D[0,:] = d
# D[:,0] = d
# D[0,0] = 0


In [ ]:
# ### Plotting the change in optimal size -


# N = range(30,100)
# OPT = np.zeros(len(N))
# LHS_HUR = np.zeros(len(N))
# LHS_HUR_ERR = np.zeros(len(N))
# L1_HUR = np.zeros(len(N))
# AVG = 100
# start_time = time.time()
# for i,n in enumerate(N):
#     if i % 5 == 0:
#         print("Completed:" + str(i/len(N)*100) + "%")
#         print("Elapsed Time: " + str((time.time() - start_time)/60))
#     samples= list()
#     for j in range(AVG): # average over 10 instances
#         p = np.power(n,-.5)
#         B = False
#         while B == False:
#             G = random_geometric_weighted_graph(n,p)
#             B = G.is_connected()
#             if B:
#                 (H,M,S2) = left_edge_heuristic(G)
#                 if len(S2) > 0: # Make sure the graph is actually broken
#                     samples.append(len(S2))
# #                     print(samples)
#                     LHS_HUR[i] += len(S2)/AVG
#                     #Todo: add error bars!! maybe np.std/sqrt(AVG)
# #                     S1 = l1_min_heuristic(G)
# #                     L1_HUR[i] += len(S1)/AVG
# #                     OPT[i] += len(brute_force_IOMR(G))/AVG # Solve hitting set, takes yearsssss
#                 else:
#                     B = False # if it's not broken - resample (and again, make sure that it's connected)
#         LHS_HUR[i] = sum(samples)/AVG
#         LHS_HUR_ERR[i] = np.std(samples)/np.sqrt(AVG)
# print("###Total Time ###")
# print((time.time() - start_time)/60)

    
    

In [522]:
def experiment_change_in_optimal_size(n1=30,n2=100,p = lambda x: np.power(x,-0.5),AVG = 30):
    """
    p would be a function of n, or a constant function. But we definitly expect it to be a function N -> (0,1)
    """
    N = range(n1,n2)
    # Set up lists to collect results
    LHS_HUR = np.zeros(len(N))
    LHS_HUR_ERR = np.zeros(len(N))
    LHS_HUR_EXP_FRAC = np.zeros(len(N))
    LHS_HUR_EDGE_FRAC = np.zeros(len(N))
#     L1_HUR = np.zeros(len(N))
#     L1_HUR_ERR = np.zeros(len(N))
    PIVOT_HUR = np.zeros(len(N))
    PIVOT_HUR_ERR = np.zeros(len(N))
    PIVOT_HUR_EXP_FRAC = np.zeros(len(N))
#     PIVOT_HUR_EDGE_FRAC = np.zeros(len(N))
    
    PIVOT_COMPLETE_HUR = np.zeros(len(N))
    PIVOT_COMPLETE_HUR_ERR = np.zeros(len(N))
    PIVOT_COMPLETE_HUR_EXP_FRAC = np.zeros(len(N))
    OPT = np.zeros(len(N))
    
    start_time = time.time()
    for i,n in enumerate(N):
        ### Keep track of progress
        if i % 5 == 0:
            print("Completed:" + str(i/len(N)*100) + "% \n" + "Elapsed Time: " + str((time.time() - start_time)/60))
            
        ## Keep the size of hitting sets over all iterations of the same graph size.
        LHS_samples= list()
        PIVOT_samples=list()
        PIVOT_COMPLETE_samples=list()
        OPT_samples=list()
#         L1_samples= list()

        for j in range(AVG): # average over AVG instances
            B = False #Flag if current instance is connected
            while B == False:
                G = random_geometric_weighted_graph(n,p(n))
                ## Todo: Change this to iterate over all connected components
                B = G.is_connected()
                if B: #If the graph is connected
                    S2 = left_edge_heuristic(G) 
                    if len(S2) > 0: # Make sure the graph is actually broken
                        LHS_samples.append(len(S2)) #Save the HS size found in LHS

                        PIVOT_samples.append(len(pivot_heuristic(G))) #Save the HS size found in Pivot
                        PIVOT_COMPLETE_samples.append(len(MVD_Pivot(geometric_complete_graph(n,p(n))))) #Save the HS size found in Pivot
        
#                         OPT_samples.append(len(brute_force_IOMR(G))) #Solve actual Minimal HS for the graph
#                         S1 = l1_min_heuristic(G)
#                         L1_samples.append(len(S1))
                        
                    else:
                        B = False # if it's not connected - resample (and again, make sure that it's connected)
                
#         L1_HUR[i] += sum((L1_samples))/AVG
#         L1_HUR_ERR = np.std(L1_samples)/np.sqrt(AVG) #error bars, is this the computation? Standard error

        ## After finishing AVG iterations, we can update the result
#         OPT[i] = sum(OPT_samples)/AVG 
        LHS_HUR[i] = sum(LHS_samples)/AVG
        PIVOT_HUR[i] = sum(PIVOT_samples)/AVG
        PIVOT_COMPLETE_HUR[i] = sum(PIVOT_COMPLETE_samples)/AVG
        
        # Error Bars
        LHS_HUR_ERR[i] = np.std(LHS_samples)/np.sqrt(AVG) 
        PIVOT_HUR_ERR[i] = np.std(PIVOT_samples)/np.sqrt(AVG) #error bars, is this the computation? Standard error
        PIVOT_COMPLETE_HUR_ERR[i] = np.std(PIVOT_COMPLETE_samples)/np.sqrt(AVG) 
        
        ## Compute ||SOL||_0 as a fraction over expected number of edges
        LHS_HUR_EXP_FRAC[i] = LHS_HUR[i]/(n*(n-1)*p(n)/2)
        PIVOT_HUR_EXP_FRAC[i] = PIVOT_HUR[i]/(n*(n-1)*p(n)/2)
        PIVOT_COMPLETE_HUR_EXP_FRAC[i] = PIVOT_COMPLETE_HUR[i]/(n*(n-1)*p(n)/2)
        ## TODO: Compute ||SOL||_0 as a fraction over the empirical number of edges. Need deeper in the loops

    return LHS_HUR, LHS_HUR_ERR, LHS_HUR_EXP_FRAC, PIVOT_HUR, PIVOT_HUR_ERR, PIVOT_HUR_EXP_FRAC, PIVOT_COMPLETE_HUR, PIVOT_COMPLETE_HUR_ERR, PIVOT_COMPLETE_HUR_EXP_FRAC 
#     return LHS_HUR,PIVOT_HUR,OPT

# TEST
list(enumerate(np.linspace(.1,.9,10)))

[(0, 0.1),
 (1, 0.18888888888888888),
 (2, 0.2777777777777778),
 (3, 0.3666666666666667),
 (4, 0.4555555555555556),
 (5, 0.5444444444444445),
 (6, 0.6333333333333333),
 (7, 0.7222222222222222),
 (8, 0.8111111111111111),
 (9, 0.9)]

In [525]:
# N1=10
# N2= 15
# ## Test With Hitting set
# TEST = experiment_change_in_optimal_size(N1,N2,lambda x: 1/np.sqrt(x),AVG = 30)

for i,p in enumerate(np.linspace(.5,.9,2)):
        print("#####")
        print("p = " + str(p))
        print("#####")
        N1=30
        N2=100
        N = range(N1,N2)
        SOLNS = experiment_change_in_optimal_size(N1,N2,lambda x: p,AVG=20)
        plt.plot(N,SOLNS[0],label = "LHS Heuristic")
        plt.plot(N,SOLNS[3],label = "Pivot Algorithm")
        plt.plot(N,SOLNS[6],label = "Pivot (complete graph)")
        plt.plot(N,[n*(n-1)/2*p for n in N],label = "expected # edges", alpha=.2)
        plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
        plt.legend()
        plt.title("p = " + str(p))
        plt.savefig(str(p)+".png")
        plt.clf()
        
        ## Same plot with error bars
        plt.errorbar(N,SOLNS[0],yerr=SOLNS[1],ecolor = 'red',label = "LHS Heuristic")
        plt.errorbar(N,SOLNS[3],yerr=SOLNS[4],ecolor = 'red',label = "Pivot Algorithm")
        plt.errorbar(N,SOLNS[6],yerr=SOLNS[7],ecolor = 'red',label = "Pivot (complete graph)")
        plt.plot(N,[n*(n-1)/2*p for n in N],label = "expected # edges", alpha=.2)
        plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
        plt.legend()
        plt.title("p = " + str(p))
        plt.savefig(str(p)+"errorbar.png")
        plt.clf()
        
        ## Plot the fraction w.r.t expected number of edeges
        plt.plot(N,SOLNS[2],label = "LHS Heuristic")
        plt.plot(N,SOLNS[5],label = "Pivot Algorithm")
        plt.plot(N,SOLNS[8],label = "Pivot (complete graph)")
        plt.plot(N,[n*(n-1)/2*p for n in N],label = "expected # edges", alpha=.2)
        plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
        plt.legend()
        plt.title("p = " + str(p))
        plt.savefig(str(p)+"expectation_fraction.png")
        plt.clf()

#####
p = 0.5
#####
Completed:0.0% 
Elapsed Time: 6.159146626790365e-07
Completed:7.142857142857142% 
Elapsed Time: 0.1590123176574707
Completed:14.285714285714285% 
Elapsed Time: 0.41837343374888103
Completed:21.428571428571427% 
Elapsed Time: 0.8099832018216451
Completed:28.57142857142857% 
Elapsed Time: 1.390824548403422
Completed:35.714285714285715% 
Elapsed Time: 2.2041900316874186
Completed:42.857142857142854% 
Elapsed Time: 3.304131515820821
Completed:50.0% 
Elapsed Time: 4.761102334658305
Completed:57.14285714285714% 
Elapsed Time: 6.673762249946594
Completed:64.28571428571429% 
Elapsed Time: 9.152233366171519
Completed:71.42857142857143% 
Elapsed Time: 12.281547466913858
Completed:78.57142857142857% 
Elapsed Time: 16.23350261449814
Completed:85.71428571428571% 
Elapsed Time: 21.14459508260091
Completed:92.85714285714286% 
Elapsed Time: 27.114714149634043
#####
p = 0.9
#####
Completed:0.0% 
Elapsed Time: 2.86102294921875e-07
Completed:7.142857142857142% 
Elapsed Time: 0.2513310

KeyboardInterrupt: 

<Figure size 432x288 with 0 Axes>

In [ ]:
# plt.errorbar(range(30,100),TEST[0],yerr=TEST[1],ecolor = 'red', label = str(1))
# plt.errorbar(range(30,100),TEST[2],yerr=TEST[3],ecolor = 'red', label = str(2))
N = range(N1,N2)
plt.plot(N,TEST[0], label = "LHS Heuristic")
plt.plot(N,TEST[3],label = "Pivot Algorithm")
plt.plot(N,[np.log(n) for n in N])
plt.plot(N,[np.sqrt(n) for n in N])
plt.legend()
plt.title(exp_dict[0][0])
plt.show()

exp_dict = {0:("$p = 1/\\sqrt{n}}$",lambda x: 1/np.sqrt(x)),
                1:("p = .2",lambda x: 0.2),
                2:("$p = .7$",lambda x: 0.7),
                2:("$p = .7$",lambda x: 0.7),
                3:("$p = n^{-1/3}$",lambda x : np.power(x,-(1/3))),
                4:("$p = n^{-1/4}$",lambda x : np.power(x,-(1/4))) ,   
                4:("$p = 1/log(n)$",lambda x : np.log(x)),
                5:("$p = 1/log(n)$",lambda x : np.log(x)),
                }


In [ ]:
# N1=30
# N2=32
exp_dict = {0:("$p = 1/\\sqrt{n}}$",lambda x: 1/np.sqrt(x)),
                1:("$p = .2$",lambda x: 0.2),
                2:("$p = .7$",lambda x: 0.7),
                2:("$p = .7$",lambda x: 0.7),
                3:("$p = n^{-1/3}$",lambda x : np.power(x,-(1/3))),
                4:("$p = n^{-1/4}$",lambda x : np.power(x,-(1/4))) ,   
                5:("$p = 1/\\log(n)$",lambda x : 1/np.log(x)),
                }
for i in range(6):
    print("#####")
    print(exp_dict[i][0])
    print("#####")
    N1=30
    N2=100
    N = range(N1,N2)
    SOLNS = experiment_change_in_optimal_size(N1,N2,exp_dict[i][1],AVG=40)
    plt.plot(N,SOLNS[0],label = "LHS Heuristic")
    plt.plot(N,SOLNS[3],label = "Pivot Algorithm")
    plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
    plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
    plt.legend()
    plt.title(exp_dict[i][0])
    plt.savefig(str(i)+".png")
    plt.clf()
    
    ## Same plot with error bars
    plt.errorbar(N,SOLNS[0],yerr=SOLNS[1],ecolor = 'red',label = "LHS Heuristic")
    plt.errorbar(N,SOLNS[3],yerr=SOLNS[4],ecolor = 'red',label = "Pivot Algorithm")
    plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
    plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
    plt.legend()
    plt.title(exp_dict[i][0])
    plt.savefig(str(i)+"errorbar.png")
    plt.clf()
    
    ## Plot the fraction w.r.t expected number of edeges
    plt.plot(N,SOLNS[2],label = "LHS Heuristic")
    plt.plot(N,SOLNS[5],label = "Pivot Algorithm")
    plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
    plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
    plt.legend()
    plt.title(exp_dict[i][0])
    plt.savefig(str(i)+"expectation_fraction.png")
    plt.clf()

In [ ]:
### Constant p ###

print("Starting - p=0.2")
RES_2 = experiment_change_in_optimal_size(30,100,lambda x : .2)
print("Starting - p=0.7")
RES_7 = experiment_change_in_optimal_size(30,100,lambda x : .7)

# RES_CONST = list()
# P = np.linspace(.5,.6,5)
# for p in P:
#     print("starting p = " + str(p))
#     RES_CONST.append(experiment_change_in_optimal_size(30,100,lambda x : p))

# for i,L in enumerate(RES_CONST):
#     plt.plot(N,L[0],label = str(P[i]))


In [ ]:
### Different Values of p ###
print("Starting - p = n^-t where t = 1/2")
RES_12 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(1/2)),AVG=70)


In [ ]:
# RES_121 = experiment_change_in_optimal_size(30,50,lambda x : np.power(x,-(1/2)),AVG=40)
# plt.plot(range(30,50),RES_12[0])
# plt.plot(range(30,50),RES_121[0])
# plt.show()

In [ ]:
print("Starting - p = n^-t where t = 1/3")
RES_13 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(1/3)))

In [ ]:
print("Starting - p = 1/log(n)")
RES_logn = experiment_change_in_optimal_size(30,100,lambda x : np.power(np.log(x),-1))

In [ ]:
print("Starting - p = n^-t where t = 1/4")
RES_14 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(1/4)))

In [ ]:
print("Starting - p = n^-t where t = 3/5")
RES_35 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(3/5)))


In [ ]:
print("Starting - p = n^-t where t = 4/7")
RES_47 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(4/7)))


In [ ]:
print("Starting - p = n^-t where t = 5/9")
RES_59 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(5/9)))


In [ ]:
print("Starting - p = n^-t where t = 6/11")
RES_611 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(6/11)))

In [ ]:
print("Starting - p = n^-t where t = 2/3")
RES_23 = experiment_change_in_optimal_size(30,100,lambda x : np.power(x,-(2/3)))

In [ ]:
# N = range(30,100)

# ## Const p
# RES_2
# RES_7

# # p = power of 1/n or 1/log(n)
# RES_12
# RES_13
# RES_14
# RES_logn


# #p Slightly larger than 3/5
# RES_23

# # p = -k/2k-1 power of n
# RES_35
# RES_47
# RES_59
# RES_611


# PL = [RES_2, RES_7,RES_611, RES_23]
# for i,L in enumerate([RES_12]):
# #         plt.errorbar(N,L[0],yerr=L[1],ecolor = 'red', label = str(i))
# #         plt.yscale('log')
#         ## log-log scale transforms power functions to lines, where the sope is roughly the power
# #         plt.yscale('log')
# #         plt.xscale('log')
#         plt.plot(N,L[0], label = str(i))
#         plt.plot(N,[np.sqrt(n) for n in N], label = str(i))
# plt.legend()
# plt.show()


In [ ]:
# plt.plot(N,[n**5 for n in N])
# # plt.plot(N,[n**2 for n in N])
# plt.yscale('log')
# plt.xscale('log')
# plt.show()

In [ ]:
# T1 = Graph([(0, 1, 1), (0, 3, 1), (0, 4, 1), (1, 2, 2), (1, 4, 1), (2, 3, 2), (2, 4, 6), (3, 4, 1)])
# T1.show(layout="planar",edge_labels=1)
# left_edge_heuristic(T1)

In [ ]:
# T = Graph([(0, 1, 4),
#  (0, 2, 1),
#  (0, 3, 4),
#  (0, 4, 1),
#  (1, 2, 2),
#  (1, 4, 2),
#  (2, 3, 2),
#  (3, 4, 1)])
# T.show(edge_labels = 1)
# T.weighted_adjacency_matrix()[0]

In [ ]:
N = range(30,100)
plt.plot(N,TRY_7_AGAIN[0],label = "LHS Heuristic")
plt.plot(N,TRY_7_AGAIN[3],label = "Pivot Algorithm")
plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
plt.legend()
# plt.title(exp_dict[i][0])
plt.savefig("TRY_7_AGAIN.png")
plt.clf()

## Same plot with error bars
plt.errorbar(N,TRY_7_AGAIN[0],yerr=TRY_7_AGAIN[1],ecolor = 'red',label = "LHS Heuristic")
plt.errorbar(N,TRY_7_AGAIN[3],yerr=TRY_7_AGAIN[4],ecolor = 'red',label = "Pivot Algorithm")
plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
plt.legend()
# plt.title(exp_dict[i][0])
plt.savefig("TRY_7_AGAINerrorbar.png")
plt.clf()

## Plot the fraction w.r.t expected number of edeges
plt.plot(N,TRY_7_AGAIN[2],label = "LHS Heuristic")
plt.plot(N,TRY_7_AGAIN[5],label = "Pivot Algorithm")
# plt.plot(N,[n*(n-1)/2*.7 for n in N],label = "expected # edges", alpha=.2)
# plt.plot(N,[n*(n-1)/2 for n in N],label = "max # edges",alpha=.2)
plt.legend()
# plt.title(exp_dict[i][0])
plt.savefig("TRY_7_AGAINexpectation_fraction.png")
plt.clf()